# STEP 0 — Kaggle Environment Bootstrap

Run this **once** before any other notebook. It:
1. Uploads the `src/` code to `/kaggle/working/light_paper/src/`
2. Verifies GPU + VRAM available
3. Does a quick smoke test loading the smallest model

In Kaggle: add this entire `light_paper/` folder as a **Dataset** input,
then in each notebook set `BASE_DIR = Path('/kaggle/input/light-paper')` for src
and `Path('/kaggle/working/light_paper')` for outputs.

In [ ]:
# ── GPU check ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPUs available:')
print(result.stdout)

import torch
print(f'PyTorch CUDA available: {torch.cuda.is_available()}')
print(f'Device count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB')

In [ ]:
# ── Directory setup ───────────────────────────────────────────────────────
import sys, shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working/light_paper')
SRC_DIR  = WORK_DIR / 'src'
DATA_DIR = WORK_DIR / 'data'
RES_DIR  = WORK_DIR / 'results'
FIG_DIR  = WORK_DIR / 'figures'

for d in [SRC_DIR, DATA_DIR, RES_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy src from input dataset (if running from Kaggle dataset input)
INPUT_SRC = Path('/kaggle/input/light-paper/src')
if INPUT_SRC.exists():
    shutil.copytree(INPUT_SRC, SRC_DIR, dirs_exist_ok=True)
    print(f'✓ Copied src/ from {INPUT_SRC}')
else:
    print(f'ℹ src/ not found at {INPUT_SRC} — add light_paper as a Kaggle Dataset input')

sys.path.insert(0, str(WORK_DIR))
print(f'✓ sys.path configured')

In [ ]:
# ── Smoke test: load smallest model ───────────────────────────────────────
from src.models import get_model_by_name, load_model
from src.prompts import make_accuracy_prompt
from src.inference import generate_responses, format_messages

config = get_model_by_name('Qwen2.5-0.5B')
model, tokenizer = load_model(config)

test_prompt = make_accuracy_prompt('What is 2 + 2?', 'direct', 'gsm8k')
responses = generate_responses(model, tokenizer, [test_prompt], max_new_tokens=64, batch_size=1)

print(f'Test response: {responses[0]}')
print('\n✅ Smoke test passed — everything is working')

import gc
del model, tokenizer
gc.collect()
import torch
torch.cuda.empty_cache()
print('GPU freed — ready to run Step 1')